In [1]:
from selenium import webdriver
from selenium.webdriver.support.ui import Select

from selenium.common.exceptions import WebDriverException, NoSuchElementException, TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
pd.set_option('display.max_columns', None)
#from sqlalchemy import create_engine
import sqlite3

In [2]:
conn = sqlite3.connect('nba_data.db')

# GAME DATES

In [3]:
season = "2024-25"

In [4]:
from nba_api.stats.endpoints import LeagueGameFinder

# Set parameters for the desired season and team
gamefinder = LeagueGameFinder(season_nullable=season,season_type_nullable="Regular Season")

# Get games as a DataFrame
games = gamefinder.get_data_frames()[0]

# Optional: Filter to only include date, home and away teams, and matchup info
schedule = pd.DataFrame(games[['GAME_DATE']])

# Convert game date to a readable format if needed
schedule['GAME_DATE'] = pd.to_datetime(schedule['GAME_DATE']).dt.strftime('%m/%d/%Y')

In [5]:
dates = schedule.drop_duplicates()

In [6]:
existing_dates = pd.read_sql(f"SELECT GAME_DATE FROM shotclock_data", conn)

unique_dates = dates.merge(
    existing_dates, 
    on="GAME_DATE", 
    how="left", 
    indicator=True
)

# Keep only the dates from `dates` that do not exist in `existing_dates`
unique_dates = unique_dates[unique_dates["_merge"] == "left_only"]

# Drop the '_merge' column if you don't need it
unique_dates = unique_dates.drop(columns=["_merge"])

In [7]:
#unique_dates

# SHOT DATA 

## SHOT CLOCK DATA

In [8]:
from nba_api.stats.endpoints import LeagueDashPlayerPtShot

In [9]:
# Initialize an empty DataFrame to collect all tracking data
tracking_df_total = pd.DataFrame()

# Filter options for tracking
filter_options = ['24-22','22-18 Very Early','18-15 Early','15-7 Average', '7-4 Late', '4-0 Very Late']

# Initial list of dates to process
dates_new = unique_dates[['GAME_DATE']].copy()  # Reference to the main dates DataFrame

# Dictionary to log failed dates and filter options
failed_log = {}

# Loop to fetch data until all dates are processed
while not dates_new.empty:
    all_data = []  # Temporary storage for each iteration
    print(f"Starting batch with {len(dates_new)} dates left to process.")

    # Process the first 20 dates in dates_new
    batch_dates = dates_new['GAME_DATE'].head(20).tolist()  # Get the first 20 dates as a list

    # Loop through each date and filter option
    for date in batch_dates:
        for tracking in filter_options:
            attempt = 0
            success = False

            while attempt < 3 and not success:
                try:
                    shot_data = LeagueDashPlayerPtShot(
                        season=season,
                        per_mode_simple='PerGame',
                        shot_clock_range_nullable=tracking,
                        date_from_nullable=date,
                        date_to_nullable=date
                    )
                    temp_df = shot_data.get_data_frames()[0]
                    temp_df['GAME_DATE'] = date
                    temp_df['SHOT_CLOCK_RANGE'] = tracking
                    
                    all_data.append(temp_df)
                    success = True
                    
                    time.sleep(4)  # Wait time between successful requests
                except Exception as e:
                    attempt += 1
                    print(f"Attempt {attempt} failed for date {date} and tracking {tracking}: {e}")
                    time.sleep(5 * attempt)

            # Log failures if unsuccessful after all attempts
            if not success:
                print(f"Failed to retrieve data for date {date} and tracking {tracking} after 3 attempts.")
                failed_log.setdefault(date, []).append(tracking)  # Log the failed filter for the date

    # Combine current batch into a single DataFrame
    tracking_df = pd.concat(all_data, ignore_index=True)
    tracking_df["SEASON_YEAR"] = season
    # Append new data to the total DataFrame
    tracking_df_total = pd.concat([tracking_df_total, tracking_df], ignore_index=True)

    # Remove processed dates from `dates_new`
    dates_new = dates_new[~dates_new['GAME_DATE'].isin(batch_dates)]

    # Display the progress
    print(f"Batch completed. {len(dates_new)} dates remaining.")
    time.sleep(15)

# Display or process `tracking_df_total`
print("Data retrieval complete.")

# Log the failed dates and tracking options
if failed_log:
    print("Failed dates and their tracking options:")
    for date, filters in failed_log.items():
        print(f"Date: {date}, Failed filters: {filters}")
else:
    print("All dates processed successfully.")

Starting batch with 31 dates left to process.
Batch completed. 11 dates remaining.
Starting batch with 11 dates left to process.
Batch completed. 0 dates remaining.
Data retrieval complete.
All dates processed successfully.


In [10]:
shotclock_upload = tracking_df_total

In [11]:
shotclock_upload['id'] = shotclock_upload['PLAYER_ID'].astype(str) +"_"+shotclock_upload['GAME_DATE'].astype(str)+"_"+shotclock_upload['SHOT_CLOCK_RANGE'].astype(str)

In [12]:
#shotclock_upload

In [13]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "shotclock_data"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

In [14]:
#shot_clock_data.to_sql(table_name, conn, if_exists='replace', index=False)

In [15]:
# Step 2: Create the table if it doesn't exist
if not table_exists:
    shotclock_upload.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = shotclock_upload[~shotclock_upload['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'shotclock_data' already exists. Checking for new records...
Inserted 17494 new records into 'shotclock_data'.


In [16]:
# CLOSEST DEFENDER

In [17]:
conn = sqlite3.connect('nba_data.db')
existing_dates = pd.read_sql(f"SELECT GAME_DATE FROM closestdefender_data", conn)

unique_dates = dates.merge(
    existing_dates, 
    on="GAME_DATE", 
    how="left", 
    indicator=True
)

# Keep only the dates from `dates` that do not exist in `existing_dates`
unique_dates = unique_dates[unique_dates["_merge"] == "left_only"]

# Drop the '_merge' column if you don't need it
unique_dates = unique_dates.drop(columns=["_merge"])

In [18]:
# Initialize an empty DataFrame to collect all tracking data
tracking_df_total = pd.DataFrame()

# Filter options for tracking
filter_options = ['0-2 Feet - Very Tight','2-4 Feet - Tight', '4-6 Feet - Open', '6+ Feet - Wide Open']

# Initial list of dates to process
dates_new = unique_dates[['GAME_DATE']].copy()  # Reference to the main dates DataFrame

# Dictionary to log failed dates and filter options
failed_log = {}

# Loop to fetch data until all dates are processed
while not dates_new.empty:
    all_data = []  # Temporary storage for each iteration
    print(f"Starting batch with {len(dates_new)} dates left to process.")

    # Process the first 20 dates in dates_new
    batch_dates = dates_new['GAME_DATE'].head(20).tolist()  # Get the first 20 dates as a list

    # Loop through each date and filter option
    for date in batch_dates:
        for tracking in filter_options:
            attempt = 0
            success = False

            while attempt < 3 and not success:
                try:
                    shot_data = LeagueDashPlayerPtShot(
                        season=season,
                        per_mode_simple='PerGame',
                        close_def_dist_range_nullable=tracking,
                        date_from_nullable=date,
                        date_to_nullable=date
                    )
                    temp_df = shot_data.get_data_frames()[0]
                    temp_df['GAME_DATE'] = date
                    temp_df['CLOSEST_DEFENDER_DISTANCE_RANGE'] = tracking
                    
                    all_data.append(temp_df)
                    success = True
                    
                    time.sleep(4)  # Wait time between successful requests
                except Exception as e:
                    attempt += 1
                    print(f"Attempt {attempt} failed for date {date} and tracking {tracking}: {e}")
                    time.sleep(5 * attempt)

            # Log failures if unsuccessful after all attempts
            if not success:
                print(f"Failed to retrieve data for date {date} and tracking {tracking} after 3 attempts.")
                failed_log.setdefault(date, []).append(tracking)  # Log the failed filter for the date

    # Combine current batch into a single DataFrame
    tracking_df = pd.concat(all_data, ignore_index=True)
    tracking_df["SEASON_YEAR"] = season
    # Append new data to the total DataFrame
    tracking_df_total = pd.concat([tracking_df_total, tracking_df], ignore_index=True)

    # Remove processed dates from `dates_new`
    dates_new = dates_new[~dates_new['GAME_DATE'].isin(batch_dates)]

    # Display the progress
    print(f"Batch completed. {len(dates_new)} dates remaining.")
    time.sleep(15)

# Display or process `tracking_df_total`
print("Data retrieval complete.")

# Log the failed dates and tracking options
if failed_log:
    print("Failed dates and their tracking options:")
    for date, filters in failed_log.items():
        print(f"Date: {date}, Failed filters: {filters}")
else:
    print("All dates processed successfully.")

Starting batch with 31 dates left to process.
Attempt 1 failed for date 03/28/2025 and tracking 4-6 Feet - Open: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Batch completed. 11 dates remaining.
Starting batch with 11 dates left to process.
Attempt 1 failed for date 03/23/2025 and tracking 0-2 Feet - Very Tight: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))
Batch completed. 0 dates remaining.
Data retrieval complete.
All dates processed successfully.


In [19]:
closest_defender_df = tracking_df_total

In [20]:
closest_defender_df['id'] = closest_defender_df['PLAYER_ID'].astype(str) +"_"+closest_defender_df['GAME_DATE'].astype(str)+"_"+closest_defender_df['CLOSEST_DEFENDER_DISTANCE_RANGE'].astype(str)

In [21]:
#closest_defender_df

In [22]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "closestdefender_data"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

In [23]:
# Step 2: Create the table if it doesn't exist
if not table_exists:
    closest_defender_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = closest_defender_df[~closest_defender_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'closestdefender_data' already exists. Checking for new records...
Inserted 14440 new records into 'closestdefender_data'.


In [24]:
conn = sqlite3.connect('nba_data.db')
existing_dates = pd.read_sql(f"SELECT GAME_DATE FROM dribbles_shot_data", conn)

unique_dates = dates.merge(
    existing_dates, 
    on="GAME_DATE", 
    how="left", 
    indicator=True
)

# Keep only the dates from `dates` that do not exist in `existing_dates`
unique_dates = unique_dates[unique_dates["_merge"] == "left_only"]

# Drop the '_merge' column if you don't need it
unique_dates = unique_dates.drop(columns=["_merge"])

In [25]:
# Initialize an empty DataFrame to collect all tracking data
tracking_df_total = pd.DataFrame()

# Filter options for tracking
filter_options = ['0 Dribbles','1 Dribble', '2 Dribbles', '3-6 Dribbles','7+ Dribbles']

# Initial list of dates to process
dates_new = unique_dates[['GAME_DATE']].copy()  # Reference to the main dates DataFrame

# Dictionary to log failed dates and filter options
failed_log = {}

# Loop to fetch data until all dates are processed
while not dates_new.empty:
    all_data = []  # Temporary storage for each iteration
    print(f"Starting batch with {len(dates_new)} dates left to process.")

    # Process the first 20 dates in dates_new
    batch_dates = dates_new['GAME_DATE'].head(20).tolist()  # Get the first 20 dates as a list

    # Loop through each date and filter option
    for date in batch_dates:
        for tracking in filter_options:
            attempt = 0
            success = False

            while attempt < 3 and not success:
                try:
                    shot_data = LeagueDashPlayerPtShot(
                        season=season,
                        per_mode_simple='PerGame',
                        dribble_range_nullable=tracking,
                        date_from_nullable=date,
                        date_to_nullable=date
                    )
                    temp_df = shot_data.get_data_frames()[0]
                    temp_df['GAME_DATE'] = date
                    temp_df['DRIBBLES'] = tracking
                    
                    all_data.append(temp_df)
                    success = True
                    
                    time.sleep(4)  # Wait time between successful requests
                except Exception as e:
                    attempt += 1
                    print(f"Attempt {attempt} failed for date {date} and tracking {tracking}: {e}")
                    time.sleep(5 * attempt)

            # Log failures if unsuccessful after all attempts
            if not success:
                print(f"Failed to retrieve data for date {date} and tracking {tracking} after 3 attempts.")
                failed_log.setdefault(date, []).append(tracking)  # Log the failed filter for the date

    # Combine current batch into a single DataFrame
    tracking_df = pd.concat(all_data, ignore_index=True)
    tracking_df["SEASON_YEAR"] = season
    # Append new data to the total DataFrame
    tracking_df_total = pd.concat([tracking_df_total, tracking_df], ignore_index=True)

    # Remove processed dates from `dates_new`
    dates_new = dates_new[~dates_new['GAME_DATE'].isin(batch_dates)]

    # Display the progress
    print(f"Batch completed. {len(dates_new)} dates remaining.")
    time.sleep(15)

# Display or process `tracking_df_total`
print("Data retrieval complete.")

# Log the failed dates and tracking options
if failed_log:
    print("Failed dates and their tracking options:")
    for date, filters in failed_log.items():
        print(f"Date: {date}, Failed filters: {filters}")
else:
    print("All dates processed successfully.")

Starting batch with 31 dates left to process.
Batch completed. 11 dates remaining.
Starting batch with 11 dates left to process.
Batch completed. 0 dates remaining.
Data retrieval complete.
All dates processed successfully.


In [26]:
dribbles_shot_df = tracking_df_total

In [27]:
dribbles_shot_df['id'] = dribbles_shot_df['PLAYER_ID'].astype(str) +"_"+dribbles_shot_df['GAME_DATE'].astype(str)+"_"+dribbles_shot_df['DRIBBLES'].astype(str)

In [28]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "dribbles_shot_data"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

In [29]:
# Step 2: Create the table if it doesn't exist
if not table_exists:
    dribbles_shot_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = dribbles_shot_df[~dribbles_shot_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'dribbles_shot_data' already exists. Checking for new records...
Inserted 15396 new records into 'dribbles_shot_data'.


In [30]:
conn = sqlite3.connect('nba_data.db')
existing_dates = pd.read_sql(f"SELECT GAME_DATE FROM touchtime_shot_data", conn)

unique_dates = dates.merge(
    existing_dates, 
    on="GAME_DATE", 
    how="left", 
    indicator=True
)

# Keep only the dates from `dates` that do not exist in `existing_dates`
unique_dates = unique_dates[unique_dates["_merge"] == "left_only"]

# Drop the '_merge' column if you don't need it
unique_dates = unique_dates.drop(columns=["_merge"])

In [31]:
# Initialize an empty DataFrame to collect all tracking data
tracking_df_total = pd.DataFrame()

# Filter options for tracking
filter_options = ['Touch < 2 Seconds','Touch 2-6 Seconds', 'Touch 6+ Seconds']

# Initial list of dates to process
dates_new = unique_dates[['GAME_DATE']].copy()  # Reference to the main dates DataFrame

# Dictionary to log failed dates and filter options
failed_log = {}

# Loop to fetch data until all dates are processed
while not dates_new.empty:
    all_data = []  # Temporary storage for each iteration
    print(f"Starting batch with {len(dates_new)} dates left to process.")

    # Process the first 20 dates in dates_new
    batch_dates = dates_new['GAME_DATE'].head(20).tolist()  # Get the first 20 dates as a list

    # Loop through each date and filter option
    for date in batch_dates:
        for tracking in filter_options:
            attempt = 0
            success = False

            while attempt < 3 and not success:
                try:
                    shot_data = LeagueDashPlayerPtShot(
                        season=season,
                        per_mode_simple='PerGame',
                        touch_time_range_nullable=tracking,
                        date_from_nullable=date,
                        date_to_nullable=date
                    )
                    temp_df = shot_data.get_data_frames()[0]
                    temp_df['GAME_DATE'] = date
                    temp_df['TOUCH_TIME'] = tracking
                    
                    all_data.append(temp_df)
                    success = True
                    
                    time.sleep(4)  # Wait time between successful requests
                except Exception as e:
                    attempt += 1
                    print(f"Attempt {attempt} failed for date {date} and tracking {tracking}: {e}")
                    time.sleep(5 * attempt)

            # Log failures if unsuccessful after all attempts
            if not success:
                print(f"Failed to retrieve data for date {date} and tracking {tracking} after 3 attempts.")
                failed_log.setdefault(date, []).append(tracking)  # Log the failed filter for the date

    # Combine current batch into a single DataFrame
    tracking_df = pd.concat(all_data, ignore_index=True)
    tracking_df["SEASON_YEAR"] = season
    
    # Append new data to the total DataFrame
    tracking_df_total = pd.concat([tracking_df_total, tracking_df], ignore_index=True)

    # Remove processed dates from `dates_new`
    dates_new = dates_new[~dates_new['GAME_DATE'].isin(batch_dates)]

    # Display the progress
    print(f"Batch completed. {len(dates_new)} dates remaining.")
    time.sleep(15)

# Display or process `tracking_df_total`
print("Data retrieval complete.")

# Log the failed dates and tracking options
if failed_log:
    print("Failed dates and their tracking options:")
    for date, filters in failed_log.items():
        print(f"Date: {date}, Failed filters: {filters}")
else:
    print("All dates processed successfully.")

Starting batch with 31 dates left to process.
Batch completed. 11 dates remaining.
Starting batch with 11 dates left to process.
Attempt 1 failed for date 03/21/2025 and tracking Touch 2-6 Seconds: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Batch completed. 0 dates remaining.
Data retrieval complete.
All dates processed successfully.


In [32]:
touchtime_shot_df = tracking_df_total

In [33]:
touchtime_shot_df['id'] = touchtime_shot_df['PLAYER_ID'].astype(str) +"_"+touchtime_shot_df['GAME_DATE'].astype(str)+"_"+touchtime_shot_df['TOUCH_TIME'].astype(str)

In [34]:
#touchtime_shot_df

In [35]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "touchtime_shot_data"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

In [36]:
#touchtime_shot_df.to_sql(table_name, conn, if_exists='replace', index=False)

In [37]:
# Step 2: Create the table if it doesn't exist
if not table_exists:
    touchtime_shot_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = touchtime_shot_df[~touchtime_shot_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'touchtime_shot_data' already exists. Checking for new records...
Inserted 11021 new records into 'touchtime_shot_data'.
